# BM25 on real text

Issue: #20 (optional side notebook)

## Question

On OTTO, BM25 only sees made-up category names and behaves like a random ranker (#19). Is that a fault of BM25 or of the data? Here the same BM25 runs on a real text retrieval task, where it is supposed to work.

## Data used

**SciFact** from the BEIR benchmark: 5,183 scientific abstracts (title + text), 300 test claims as queries, and human relevance judgements (each claim has one or a few supporting abstracts). Downloaded from the BEIR project page into `data/text/` (not committed; SciFact is licensed for non-commercial use, see the BEIR page). This notebook is independent of the OTTO pipeline: nothing in it is imported elsewhere.

Code: our BM25 from `notebooks/bm25.py` (#19) and the NDCG from `notebooks/metrics.py` (#13), compared with the `rank_bm25` library. Both use the **same tokeniser** (lowercase letters and digits), so differences come only from the scoring code.

In [ ]:
import io
import json
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import polars as pl
from rank_bm25 import BM25Okapi

sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")
from bm25 import BM25, tokenize
from bootstrap import bootstrap_ci, paired_bootstrap_diff
from metrics import ndcg_at_k, recall_at_k

DATA = Path("data") if Path("data").exists() else Path("../data")
TEXT = DATA / "text"
TEXT.mkdir(parents=True, exist_ok=True)
URL = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip"
if not (TEXT / "scifact" / "corpus.jsonl").exists():
    print("downloading", URL)
    with urllib.request.urlopen(URL, timeout=120) as r:
        zipfile.ZipFile(io.BytesIO(r.read())).extractall(TEXT)
root = TEXT / "scifact"
print(sorted(p.name for p in root.iterdir()))

In [ ]:
corpus = [json.loads(line) for line in (root / "corpus.jsonl").read_text().splitlines()]
queries = {q["_id"]: q["text"] for q in (json.loads(line) for line in (root / "queries.jsonl").read_text().splitlines())}
qrels = pl.read_csv(root / "qrels" / "test.tsv", separator="\t")
relevant = {}
for qid, did, score in qrels.iter_rows():
    if score > 0:
        relevant.setdefault(str(qid), set()).add(str(did))
test_ids = sorted(relevant)
doc_ids = [d["_id"] for d in corpus]
docs = [tokenize((d.get("title") or "") + " " + (d.get("text") or "")) for d in corpus]
print(f"{len(corpus):,} documents (mean length {np.mean([len(d) for d in docs]):.0f} tokens), {len(test_ids)} test queries, {sum(len(v) for v in relevant.values())} relevant pairs")
assert len(test_ids) == 300

## Score with our BM25 and with the library

In [ ]:
t = time.time()
ours = BM25(docs, k1=1.5, b=0.75)
lib = BM25Okapi(docs, k1=1.5, b=0.75)
print(f"indexes built in {time.time() - t:.1f} s")

rows = []
max_diff = 0.0
t = time.time()
for qid in test_ids:
    q = tokenize(queries[qid])
    s_ours = ours.scores(q)
    s_lib = lib.get_scores(q)
    max_diff = max(max_diff, float(np.abs(s_ours - s_lib).max()))
    labels = {"click": relevant[qid]}                         # one relevance grade: a relevant abstract has gain 1
    top_ours = [doc_ids[i] for i in np.argsort(-s_ours, kind="stable")[:10]]
    top_lib = [doc_ids[i] for i in np.argsort(-s_lib, kind="stable")[:10]]
    rows.append({"qid": qid, "ours_ndcg10": ndcg_at_k(top_ours, labels, k=10), "lib_ndcg10": ndcg_at_k(top_lib, labels, k=10),
                 "ours_recall100": recall_at_k([doc_ids[i] for i in np.argsort(-s_ours, kind="stable")[:100]], relevant[qid], k=100)})
res = pl.DataFrame(rows)
print(f"scored {len(test_ids)} queries in {time.time() - t:.0f} s; largest score difference between the two implementations: {max_diff:.2e}")

In [ ]:
o = bootstrap_ci(res["ours_ndcg10"].to_numpy(), seed=0)
l = bootstrap_ci(res["lib_ndcg10"].to_numpy(), seed=0)
d = paired_bootstrap_diff(res["ours_ndcg10"].to_numpy(), res["lib_ndcg10"].to_numpy(), seed=0)
r100 = bootstrap_ci(res["ours_recall100"].to_numpy(), seed=0)
print(pl.DataFrame({"implementation": ["our BM25 (#19)", "rank_bm25 library"], "NDCG@10": [round(o[0], 4), round(l[0], 4)], "95% CI": [f"[{o[1]:.4f}, {o[2]:.4f}]", f"[{l[1]:.4f}, {l[2]:.4f}]"]}))
print(f"paired difference (ours minus library): {d[0]:+.5f} [{d[1]:+.5f}, {d[2]:+.5f}]; our recall@100: {r100[0]:.4f} [{r100[1]:.4f}, {r100[2]:.4f}]")
assert abs(o[0] - l[0]) < 0.01, "our BM25 and the library differ by more than 0.01 NDCG@10"
assert max_diff < 1e-6

## What it shows

| Implementation | NDCG@10 (300 test claims) | 95% CI |
|---|---|---|
| our BM25 (#19) | 0.6523 | [0.6042, 0.6967] |
| `rank_bm25` library | 0.6523 | [0.6042, 0.6967] |

Our recall@100 is 0.873. The two implementations agree to 1e-13 on every score (largest difference 2.8e-14), so the paired NDCG@10 difference is exactly 0 and well inside the required 0.01. With a search index of 5,183 abstracts, both were built and scored in a few seconds.

**The lesson.** On real text BM25 works well: it puts a relevant abstract in the top 10 for most claims (NDCG@10 0.65; published BEIR results for BM25 on SciFact are in the same range, in the mid 0.6s, quoted from memory and not re-checked here). On the OTTO data (#19) the same code scored NDCG@10 of 0.0007, no better than a random order. The method is the same, so the difference is the **data**: OTTO items have no text, and synthetic category names give BM25 nothing to separate items inside a category. The near-random OTTO result is a limit of that data, not a verdict on BM25.

The SciFact data are not committed and nothing here is used by the OTTO pipeline.

## What I learned

To be written by the owner of the project.